# 전처리 재점검
원본은 수정하지 않습니다. 이전 노트북의 오류 조사 셀을 모두 실행하는 대신 이 순서를 사용합니다.
공식 기준: 손목 ECG 500Hz / PPG 100Hz. 1단계는 구조 검사, 2단계는 기존 학습 파일 검사입니다. 통과해도 손목 라벨 의미나 임상 성능이 확정되는 것은 아닙니다. 이 노트북은 학습하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, json, torch
assert torch.cuda.is_available(), 'GPU 런타임으로 연결하세요'
BASE=Path('/content/drive/MyDrive/Capdyssey/af_integrated')
assert (BASE/'raw/longterm_af_v3').is_dir(), '원본 경로 없음'
REPO=Path('/content/capdyssey_preprocessing_audit')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/dung-ho03/capdyssey.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin','5a75db532c293e3c85ae487deef13a5a82dc37ed'],cwd=REPO,check=True)
subprocess.run(['git','checkout','5a75db5'],cwd=REPO,check=True)
def run(script):
    p=subprocess.Popen([sys.executable,'-u','scripts/'+script,'--base',str(BASE)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait()!=0: raise RuntimeError('검사 실패: 오류 수정 전 다음 단계 진행 금지')
print('연결 완료:',torch.cuda.get_device_name(0))

In [ ]:
# 1단계. 원본 39명 구조 검사: Drive 읽기로 시간이 걸릴 수 있습니다.
run('audit_preprocessing_step1.py')

In [ ]:
# 2단계. 현재 학습 파일 전체 검사. 통과 출력은 임상적 검증을 의미하지 않습니다.
run('audit_prepared_bundle.py')

In [ ]:
raw=json.loads((BASE/'audits/preprocessing_step1/report.json').read_text())
prepared=json.loads((BASE/'audits/prepared_bundle/report.json').read_text())
assert not raw['read_errors'] and not raw['missing_wrist_files']
assert prepared['structural_checks_passed'] and not prepared['errors']
print('원본 검사 인원:',len(raw['subjects']))
print('QRS 범위 이상:',[r['subject'] for r in raw['subjects'] if not r['qrs_in_bounds_one_based']])
print('검증된 학습 구간:',prepared['passed_windows'])
for source,c in prepared['sources'].items():
    print(source, '참가자',len(c['subjects']),'구간',c['windows'],'AF',c['af'],'비AF',c['non_af'])
print('구조 검사 통과. 잠정 라벨 해석 유지. 임상 정답·신호 품질·실제 정렬은 별도 검토 대상.')